# Trascrizione di audio armeno → testo

Serve a costruire un corpus di **armeno parlato** per AB1J, da confrontare con l'EANC (che è scritto e letterario).

Usa `Chillarmo/whisper-large-v3-turbo-armenian`, una versione di Whisper specializzata per l'armeno (WER ~15%).
Per contare le frequenze quel margine d'errore va bene: gli sbagli si concentrano sulle parole rare, non su quelle frequenti che ci interessano.

## Prima di iniziare (su iPad o telefono)

1. **Attiva la GPU**: menu `Runtime` → `Cambia tipo di runtime` → Acceleratore hardware: **T4 GPU**. Senza, ci mette dieci volte tanto.
2. **Disattiva il blocco schermo**: Impostazioni → Schermo → Blocco automatico → **Mai**. Se il tab va in background Colab si scollega.
3. **Tieni Safari in primo piano** mentre gira. Puoi abbassare la luminosità, non cambiare app.

Se si scollega non perdi niente: la cella di trascrizione **salta i file già fatti**. Riapri, rilancia, riprende da dove era.

## 1. Collega Google Drive

Metti gli audio in una cartella di Drive chiamata `armeno_audio` (mp3, m4a, wav, qualsiasi).
Le trascrizioni finiranno in `armeno_testi`, sempre su Drive: le puoi aprire dal telefono.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
AUDIO = Path('/content/drive/MyDrive/armeno_audio')
TESTI = Path('/content/drive/MyDrive/armeno_testi')
TESTI.mkdir(parents=True, exist_ok=True)

files = sorted([p for p in AUDIO.glob('*') if p.suffix.lower() in
                {'.mp3', '.m4a', '.wav', '.ogg', '.opus', '.mp4', '.aac', '.flac'}])
print(f'{len(files)} file audio trovati')
for p in files[:10]:
    print(' ', p.name)

## 2. Carica il modello

La prima volta scarica ~1,6 GB: qualche minuto.

In [ ]:
!pip install -q --upgrade transformers accelerate

import torch
from transformers import pipeline

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
if device == 'cpu':
    print('ATTENZIONE: nessuna GPU. Runtime → Cambia tipo di runtime → T4 GPU.')

asr = pipeline(
    'automatic-speech-recognition',
    model='Chillarmo/whisper-large-v3-turbo-armenian',
    torch_dtype=torch.float16 if device.startswith('cuda') else torch.float32,
    device=device,
    chunk_length_s=30,      # spezza l'audio lungo in blocchi da 30 secondi
    stride_length_s=5,      # sovrapposizione, per non tagliare le parole a metà
)
print('modello pronto su', device)

## 3. Trascrivi

Puoi rilanciare questa cella quante volte vuoi: i file già trascritti vengono saltati.
Ogni trascrizione viene salvata **appena finita**, non alla fine di tutto.

In [ ]:
import time

for i, p in enumerate(files, 1):
    out = TESTI / (p.stem + '.txt')
    if out.exists() and out.stat().st_size > 0:
        print(f'[{i}/{len(files)}] già fatto: {p.name}')
        continue
    t0 = time.time()
    print(f'[{i}/{len(files)}] {p.name} …', end=' ')
    try:
        res = asr(str(p), generate_kwargs={'language': 'armenian', 'task': 'transcribe'})
        out.write_text(res['text'].strip(), encoding='utf-8')
        print(f"fatto in {time.time()-t0:.0f}s — {len(res['text'].split())} parole")
    except Exception as e:
        print('ERRORE:', e)

print('\nTrascrizioni in', TESTI)

## 4. Unisci tutto in un file solo

Questo è il file da mettere nel repo AB1J (cartella `wordlists/`), così può essere lemmatizzato
con lo stesso analizzatore usato per l'EANC e confrontato con la lista scritta.

In [ ]:
import re

testi = sorted(TESTI.glob('*.txt'))
corpus = '\n'.join(p.read_text(encoding='utf-8') for p in testi)
unione = TESTI / '_corpus_parlato.txt'
unione.write_text(corpus, encoding='utf-8')

token = re.findall(r'[\u0561-\u0587\u0531-\u0556\u055E\u055B\u055C]+', corpus)
print(f'{len(testi)} trascrizioni')
print(f'{len(token):,} token armeni · {len(set(token)):,} forme distinte')
print(f'\nSalvato in: {unione}')
print('\nPer un confronto sensato con l\'EANC servono almeno ~200.000 token.')
print('Sotto i 50.000 la coda è rumore e non si può concludere granché.')

## 5. Anteprima delle frequenze

Un primo sguardo, senza lemmatizzazione. Il lavoro serio si fa nel repo.

In [ ]:
from collections import Counter

c = Counter(t.lower() for t in token)
print(f"{'forma':14}{'occorrenze':>12}{'‰':>9}")
tot = sum(c.values())
for w, n in c.most_common(30):
    print(f'{w:14}{n:>12,}{n/tot*1000:>9.2f}')